In [ ]:
import os
DATASET_PATH = '/content/drive/MyDrive/GhanaMiningPrithvi'

print("Contents of DATASET_PATH:")
print(os.listdir(DATASET_PATH))

val_dir = os.path.join(DATASET_PATH, 'validation')
print("\nContents of validation folder:")
if os.path.exists(val_dir):
    files = os.listdir(val_dir)
    print(f"Number of files: {len(files)}")
    if files:
        print("First 5 files:", files[:5])
else:
    print("Folder 'validation' does not exist!")

Contents of DATASET_PATH:
['GhanaMiningPrithvi']

Contents of validation folder:
Folder 'validation' does not exist!


In [ ]:
# ============ ZELLE 1: Drive mounten (zuerst ausführen, im Browser freigeben) ============

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Zelle 2: Pakete installieren (einmalig, 1–2 Min)
!pip install -q terratorch==0.99.7 segmentation-models-pytorch==0.3.4
!pip install -q lightning==2.4.0 albumentations==1.4.10
!pip install -q rasterio==1.3.11

In [ ]:
# Zelle 3: Daten von Drive auf lokalen Speicher kopieren (5–10 Min, einmalig)
# Danach ist das Training 5–10x schneller weil lokaler SSD statt Drive-Netzwerk.
import shutil
import os

src = '/content/drive/MyDrive/GhanaMiningPrithvi/GhanaMiningPrithvi'
dst = '/content/GhanaMiningPrithvi'

if not os.path.exists(dst):
    print("Kopiere Daten von Drive auf lokalen Speicher (5–10 Min, einmalig)...")
    shutil.copytree(src, dst)
    n_train = len(os.listdir(os.path.join(dst, 'training')))
    n_val = len(os.listdir(os.path.join(dst, 'validation')))
    print(f"Fertig: {n_train} Dateien in training/, {n_val} in validation/")
else:
    print("Daten bereits lokal vorhanden.")

Kopiere Daten von Drive auf lokalen Speicher (5–10 Min, einmalig)...
Fertig: 5966 Dateien in training/, 2574 in validation/


In [ ]:
# Zelle 4: Training
import os
import torch
from segmentation_models_pytorch.encoders import encoders as smp_encoders
import rasterio
import numpy as np
from terratorch.models import PrithviModelFactory
from terratorch.datamodules import GenericNonGeoSegmentationDataModule
from terratorch.tasks import SemanticSegmentationTask
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, RichProgressBar
from lightning.pytorch.loggers import TensorBoardLogger
import albumentations as A
from albumentations.pytorch import ToTensorV2

DATASET_PATH   = '/content/GhanaMiningPrithvi'
CHECKPOINT_DIR = '/content/drive/MyDrive/checkpoints/prithvi-v2-300'

train_dir = os.path.join(DATASET_PATH, 'training')
val_dir   = os.path.join(DATASET_PATH, 'validation')
if not os.path.isdir(train_dir):
    raise FileNotFoundError(f"Ordner nicht gefunden: {train_dir}")
if not os.path.isdir(val_dir):
    raise FileNotFoundError(f"Ordner nicht gefunden: {val_dir}")
n_train = len([f for f in os.listdir(train_dir) if f.endswith('_IMG.tif')])
n_val   = len([f for f in os.listdir(val_dir) if f.endswith('_IMG.tif')])
print(f"Training:   {n_train} Patches in {train_dir}")
print(f"Validation: {n_val} Patches in {val_dir}")
if n_train == 0 or n_val == 0:
    raise ValueError("training oder validation ist leer.")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

ghana_mining_bands = ["BLUE", "GREEN", "RED", "VNIR_5", "SWIR_1", "SWIR_2"]
means = [1473.81388377, 1703.35249650, 1696.67685941, 3832.39764247, 3156.11122121, 2226.06822112]
stds  = [ 223.43533204,  285.53613398,  413.82320306,  389.61483882,  451.49534791,  468.26765909]

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    ToTensorV2()
])

datamodule = GenericNonGeoSegmentationDataModule(
    batch_size=4,
    num_workers=2,
    train_data_root=os.path.join(DATASET_PATH, 'training'),
    val_data_root=os.path.join(DATASET_PATH, 'validation'),
    test_data_root=os.path.join(DATASET_PATH, 'validation'),
    img_grep="*_IMG.tif",
    label_grep="*_MASK.tif",
    means=means,
    stds=stds,
    num_classes=2,
    train_transform=train_transform,
    dataset_bands=ghana_mining_bands,
    output_bands=ghana_mining_bands,
    no_data_replace=0,
    no_label_replace=-1,
)

model_args = {
    "backbone": "prithvi_eo_v2_300",
    "bands": ghana_mining_bands,
    "in_channels": 6,
    "num_classes": 2,
    "pretrained": True,
    "decoder": "UperNetDecoder",
    "rescale": True,
    "backbone_num_frames": 1,
    "head_dropout": 0.1,
    "decoder_scale_modules": True,
}

task = SemanticSegmentationTask(
    model_args=model_args,
    model_factory="PrithviModelFactory",
    loss="ce",
    lr=1e-3,
    ignore_index=-1,
    optimizer="AdamW",
    optimizer_hparams={"weight_decay": 0.05},
    freeze_backbone=True,
    class_names=['Non_mining', 'Mining'],
)

datamodule.setup("fit")

checkpoint_callback = ModelCheckpoint(
    dirpath=CHECKPOINT_DIR,
    monitor=task.monitor,
    save_top_k=1,
    save_last=True,
    filename="prithvi-v2-300-{epoch:02d}-{val_loss:.4f}",
)
early_stopping = EarlyStopping(monitor=task.monitor, min_delta=0.001, patience=10)
logger = TensorBoardLogger(save_dir=CHECKPOINT_DIR, name='logs')

trainer = Trainer(
    devices=1,
    precision="16-mixed",
    callbacks=[RichProgressBar(), checkpoint_callback, early_stopping],
    logger=logger,
    max_epochs=50,
    default_root_dir=CHECKPOINT_DIR,
    log_every_n_steps=1,
    check_val_every_n_epoch=1,
)

print("Training startet (ca. 3–5 h auf T4)...")
print(f"Checkpoints: {CHECKPOINT_DIR}")
print()
trainer.fit(model=task, datamodule=datamodule)

print("\nTraining fertig. Test-Auswertung...")
res = trainer.test(model=task, datamodule=datamodule)
print("Test-Ergebnis:", res)
print("Bester Checkpoint:", checkpoint_callback.best_model_path)
print("Datei von Drive herunterladen → lokal als models/prithvi-v2-300-best.ckpt speichern.")

wxc_downscaling not installed
wxc_downscaling not installed
Training:   2983 Patches in /content/GhanaMiningPrithvi/training
Validation: 1287 Patches in /content/GhanaMiningPrithvi/validation


/usr/local/lib/python3.12/dist-packages/albumentations/core/composition.py:198: UserWarning: transforms is single transform, but a sequence is expected! Transform will be wrapped into list.
  super().__init__(transforms, p)
/usr/local/lib/python3.12/dist-packages/terratorch/models/prithvi_model_factory.py:75: UserWarning: PrithviModelFactory is deprecated. Please switch to EncoderDecoderFactory.
  warnings.warn("PrithviModelFactory is deprecated. Please switch to EncoderDecoderFactory.", stacklevel=1)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or da

Training startet (ca. 3–5 h auf T4)...
Checkpoints: /content/drive/MyDrive/checkpoints/prithvi-v2-300



/usr/local/lib/python3.12/dist-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /content/drive/MyDrive/checkpoints/prithvi-v2-300 exists and is not empty.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃   ┃ Name          ┃ Type             ┃ Params ┃ Mode  ┃
┡━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ 0 │ model         │ PixelWiseModel   │  318 M │ train │
│ 1 │ criterion     │ CrossEntropyLoss │      0 │ train │
│ 2 │ train_metrics │ MetricCollection │      0 │ train │
│ 3 │ val_metrics   │ MetricCollection │      0 │ train │
│ 4 │ test_metrics  │ ModuleList       │      0 │ train │
└───┴───────────────┴──────────────────┴────────┴───────┘

...

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



Training fertig. Test-Auswertung...


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃              Test metric               ┃              DataLoader 0              ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        test/Multiclass_Accuracy        │           0.9773683547973633           │
│        test/Multiclass_F1_Score        │           0.9773683547973633           │
│     test/Multiclass_Jaccard_Index      │           0.8476790189743042           │
│  test/Multiclass_Jaccard_Index_Micro   │           0.955738365650177            │
│               test/loss                │          0.05879969522356987           │
│     test/multiclassaccuracy_Mining     │           0.8712815642356873           │
│   test/multiclassaccuracy_Non_mining   │           0.984936535358429            │
│   test/multiclassjaccardindex_Mining   │           0.7193835973739624           │
│ test/multiclassjaccardindex_Non_mining │           0.975974440574646            │
└────────────────────────────────────────┴────────────────────────────────────────┘

Test-Ergebnis: [{'test/loss': 0.05879969522356987, 'test/Multiclass_Accuracy': 0.9773683547973633, 'test/multiclassaccuracy_Non_mining': 0.984936535358429, 'test/multiclassaccuracy_Mining': 0.8712815642356873, 'test/Multiclass_F1_Score': 0.9773683547973633, 'test/Multiclass_Jaccard_Index': 0.8476790189743042, 'test/multiclassjaccardindex_Non_mining': 0.975974440574646, 'test/multiclassjaccardindex_Mining': 0.7193835973739624, 'test/Multiclass_Jaccard_Index_Micro': 0.955738365650177}]
Bester Checkpoint: /content/drive/MyDrive/checkpoints/prithvi-v2-300/prithvi-v2-300-epoch=16-val_loss=0.0000.ckpt
Datei von Drive herunterladen → lokal als models/prithvi-v2-300-best.ckpt speichern.
